# TM-RugPull dataset initial analysis

## Data collection

In [1]:
 # Loading data from .xlsx file

import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot

file = 'data/TM-RugPull.xlsx'

data = pd.read_excel(file)

# Remove a space in the end of some column names
data.columns = data.columns.str.strip()

print(data.shape)

pd.set_option('display.max_columns', None)

data.head(5)

(1000, 27)


,Project Title,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,Sign,first deposits,Blockchain Type,Smart Contract (online),smart Contract (offline),website,x profile,class,project starting date,project end date,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2)
0,HyperVerse Token (HVT),7.650000e+00,1.500000e-01,9.100000e-06,8.000000e-07,BSC,623909,25752,9.537265e-03,4.929203e-02,166600000000100,HVT,44,POSA,sourse code,CODE,https://thehyperverse.net/index.html,https://twitter.com/HyperVerse6,scam,2022-01-27,2023-07-14,148,148,29,8,74,47
1,Fintoch,1.795000e-11,1.907000e-11,1.727000e-10,1.769000e-10,BSC,"1,492,842","147,791",2.487228e+03,4.503032e+07,34403.74594,BEP-20 TOKEN*,1,POSA,sourse code,CODE,https://web.archive.org/web/20230603123631/htt...,NaN,scam,2022-07-12,2023-06-19,820,167,4530,1590,48,4
2,Flare Token,1.955000e-03,5.519000e-04,4.158000e-04,2.838000e-04,BSC,"184,694",15173,8.901714e+14,6.695544e+17,10000000000,Flare,53,POSA,sourse code,CODE,https://pipeflare.io/,https://x.com/MetaFlareToken,scam,2021-10-24,2022-11-24,421,156,6,5,1710,1150
3,Safuu Protocol,2.070000e+02,2.110000e+02,7.000000e+01,2.400000e+01,BSC,"275,530",151979,2.457331e+10,8.139816e+14,61634066.59803,SAFUU,2,POSA,sourse code,CODE,https://safuu.com/,https://x.com/safuuxofficial,scam,2022-02-03,2022-08-13,327,323,0,0,5,4
4,SCT,2.986000e-01,1.659000e-01,1.831000e-01,1.552000e-01,BSC,8445,"6,126\n",4.252172e+06,1.410127e+05,"42,896,736.739367",SCT,52,POSA,sourse code,CODE,https://supercells.jp/en/,https://x.com/scttoken,scam,2023-02-27,2024-07-09,4990000,345000,6,3,1830,594


In [2]:
# Check the balance between classes in the whole dataset
# Source: https://note.nkmk.me/en/python-pandas-value-counts/#value_counts

class_counts = data['class'].value_counts()
class_percent = data['class'].value_counts(normalize=True) * 100

print("Counts:", class_counts.to_string(), "\nIn %:", class_percent.to_string())

Counts: class
scam      599
normal    401 
In %: class
scam      59.9
normal    40.1


In [3]:
# Check general info about data

print("\nDataset information:")
data.info()


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 27 columns):
 #   Column                                                 Non-Null Count  Dtype         
---  ------                                                 --------------  -----         
 0   Project Title                                          1000 non-null   object        
 1   MaxPrice (Quarter 1)                                   1000 non-null   float64       
 2   MaxPrice (Quarter 2)                                   1000 non-null   float64       
 3   MaxPrice (Quarter 3)                                   1000 non-null   float64       
 4   MaxPrice (Quarter 4)                                   1000 non-null   float64       
 5   Blockchain                                             1000 non-null   object        
 6   the number of Transactions                             1000 non-null   object        
 7   Token concentration ratio per holder            

In [4]:
# Analysis of blockchain distribution in data

blockchain_distribution = data['Blockchain'].value_counts(normalize=True) * 100
print(blockchain_distribution.to_string())

Blockchain
ETH        63.3
BSC        32.1
POLYGON     2.4
ARBI        1.2
FANTOM      0.4
CRONO       0.2
BASE        0.2
FTM         0.1
SNOW        0.1


In [5]:
# Analyse scam rate per chain

scam_rate_per_chain = (
    data.groupby('Blockchain')['class']
        .apply(lambda s: (s == 'scam').mean())
        .sort_values(ascending=False)
)

scam_rate_percents = (scam_rate_per_chain * 100)

print(scam_rate_percents.to_string())

Blockchain
BASE       100.000000
CRONO      100.000000
FANTOM     100.000000
FTM        100.000000
BSC         75.077882
POLYGON     62.500000
ARBI        58.333333
ETH         51.658768
SNOW         0.000000


##### As far as FANTOM, CRONO, BASE and SNOW blockchains have very few samples (less than 10) and are biased (only scam tokens for BASE, CRONO and FANTOM and only normal samples for SNOW), these blockchains will be removed from the dataset to avoid overfitting.

In [8]:
# Drop blockchains with number of samples less than 1%

min_samples = 1.0
chains_to_keep = blockchain_distribution[blockchain_distribution >= min_samples].index
chains_dropped = blockchain_distribution[blockchain_distribution < min_samples]

print("Blockchains to keep:")
print(list(chains_to_keep))

print("\nDropped blockchains:")
print(chains_dropped.to_string())

data = data[data['Blockchain'].isin(chains_to_keep)].copy()

Blockchains to keep:
['ETH', 'BSC', 'POLYGON', 'ARBI']

Dropped blockchains:
Blockchain
FANTOM    0.4
CRONO     0.2
BASE      0.2
FTM       0.1
SNOW      0.1


In [ ]:
# Dataset after dropping blockchains with small number of samples

print(data.shape)
data.head(5)

In [ ]:
# Display blockchain distribution after dropping blockchains with small number of samples

blockchain_distribution = data['Blockchain'].value_counts(normalize=True) * 100
print(blockchain_distribution.to_string())

In [10]:
# Save the dataset without dropped blockchains for further enrichment
# OpenPyXL is used here to preserve hyperlinks in the file
# Ran once to save changes

# Reference: https://openpyxl.readthedocs.io/en/3.1/tutorial.html

# from openpyxl import load_workbook
#
# workbook = load_workbook(file)
# sheet = workbook['Sheet1']
#
# headings = [c.value for c in sheet[1]]
# chain_column_idx = headings.index('Blockchain')
#
# chains_to_drop = chains_dropped.index.to_list()
#
# rows_to_drop = [row[0].row for row in sheet.iter_rows(min_row=2) if row[chain_column_idx].value in chains_to_drop]
#
# for row_num in sorted(rows_to_drop, reverse=True):
#     sheet.delete_rows(row_num, 1)
#
# workbook.save('data/TM-RugPull_filtered_chains.xlsx')

990 rows remaining


##### Create a test set and a validation test

In [ ]:
# Create a test set and a validation set from the raw data to avoid data leakage
# A split is temporal, preserving project period balance (based on start date), so that both old and new projects are presented in each split
# Validation set size is about 10,5% and test set size is about 24,5% from the whole data set

# Reference: https://stackoverflow.com/questions/45516424/sklearn-train-test-split-on-pandas-stratify-by-multiple-columns

from sklearn.model_selection import train_test_split

# Convert project starting date to datetime and extract year
data['project starting date'] = pd.to_datetime(data['project starting date'], errors='coerce')
data['project starting year'] = data['project starting date'].dt.year

# Group years into broader periods
def assign_project_period(year):
    if year < 2021:
        return 'before_2021'
    elif year in [2021, 2022, 2023]:
        return str(year)
    elif year in [2024, 2025]:
        return '2024_2025'
    else:
        return 'unknown'

data['project period'] = data['project starting year'].apply(assign_project_period)

# Define seed
seed = 7

# Combined stratification label for the full dataset (class and project period)
combined_labels = data['class'].astype(str) + '|' + data['project period'].astype(str)

# Split the data first on train set and set for test and validation
train_set, test_and_val_set = train_test_split(data, test_size=0.35, random_state=seed, stratify=combined_labels)

# Combined stratification label for test_and_val_set (class and project period)
subset_combined_labels = (test_and_val_set['class'].astype(str) + '|' + test_and_val_set['project period'].astype(str))

#Split the part for test and validation into test set and validation set
test_set, val_set = train_test_split(test_and_val_set, test_size=0.3, random_state=seed, stratify=subset_combined_labels)

# Output the shapes to verify the splits
print("Training set shape:", train_set.shape)
print("Test set shape:", test_set.shape)
print("Validation set shape:", val_set.shape)

# Create a list of sets to perform further feature engineering on all subsets of data
data_sets = [train_set, test_set, val_set]

train_set.head(5)

In [ ]:
# Verify that there is no overlap between sets, no data leak at this stage
print("Overlap between train and test:", np.intersect1d(train_set.index, test_set.index).size)
print("Overlap between train and validation:", np.intersect1d(train_set.index, val_set.index).size)
print("Overlap between test and validation:", np.intersect1d(test_set.index, val_set.index).size)

In [ ]:
# Delete columns that are not useful for further analysis

for data_set in data_sets:
    data_set.drop(columns=['Project Title', 'Sign', 'website', 'x profile', 'Smart Contract (online)', 'smart Contract (offline)', 'project starting date', 'project end date', 'project starting year', 'project period'], inplace=True)

# Check the result on train set
train_set.head(10)

## Raw data analysis

#### Performed first on raw data to get overall idea of what the dataset contains
##### All further exploratory operations except checking general info should be done only on the train set to avoid data leakage.
##### Analysis of skew and correlation is performed only on numeric features first. Categorial features are initially analysed separately, then encoded during pre-processing.

In [ ]:
# Encode class label

from sklearn.preprocessing import LabelEncoder

for data_set in data_sets:
    le = LabelEncoder()
    data_set['class'] = data_set['class'].map({'normal': 0, 'scam': 1})

train_set

In [ ]:
# Check the data description

description = train_set.describe()
description

In [ ]:
# Check the balance between classes in the training set

class_counts_train = data['class'].value_counts()
class_percent_train = data['class'].value_counts(normalize=True) * 100

print("Counts:", class_counts_train.to_string(), "\nIn %:", class_percent_train.to_string())

In [ ]:
# Check for missing values

print("\nMissing values:\n", train_set.isnull().sum())

In [ ]:
print(train_set.dtypes)

In [ ]:
# Transfer all data to numeric values

# Columns with object datatype
cols_to_clean = [
    'the number of Transactions',
    'Token concentration ratio per holder',
    'Token balance',
    'first deposits',
    'Google results for project website (first day)',
    'Google results for project website (duration/2)'
]

# Apply to all data splits
data_sets = [train_set, test_set, val_set]

for i, data_set in enumerate(data_sets):
    for col in cols_to_clean:
        data_sets[i][col] = pd.to_numeric(
            data_set[col].astype(str)
                   .str.replace('\xa0', '', regex=False)        # Remove non-breaking whitespaces
                   .str.replace(',', '', regex=False)           # Strip comas separating numeric values
                   .str.strip(),                                # Remove surrounding whitespaces
            errors='coerce'                                     # If cannot parse, put NaN
        )

train_set, test_set, val_set = data_sets

# Verify the result
print("Datatypes:\n", train_set[cols_to_clean].dtypes)
print("\nMissing values:\n", train_set[cols_to_clean].isnull().sum())
print("\nColumns description\n", train_set[cols_to_clean].describe())

In [ ]:
# Some extreme values (like 10ˆ59) result in overflow warnings when computing skew, and NaN values for 'Total Variance'
# and 'Variance of holders with more than 1% tokens'
# These values were capped at the 99.5 percentile estimated from the training data and applied to all datasets

# Reference: S. Galli, Python Feature Engineering Cookbook: Over 70 recipes for creating, engineering, and transforming features to build machine learning models, Packt Publishing, 2022.

cols_to_clip = [
    'Total Variance',
    'Variance of holders with more than 1% tokens'
]

# Clip at 99.5% quantile (computed on train_set only)
upper_q = 0.995

for col in cols_to_clip:
    upper = train_set[col].quantile(upper_q)
    for data_set in data_sets:
        data_set[col] = data_set[col].clip(upper=upper)

In [ ]:
# Check for skew for numeric columns

numeric_cols = train_set.select_dtypes(include=['number']).columns
skew = train_set[numeric_cols].skew()

skew

### Visualisation

##### Based on CSM010-2024-APR, Topic 2, Lab.2.15. Analysing data

In [ ]:
# Histograms

train_set.hist(figsize=[30, 30])
pyplot.show()

In [ ]:
# Density plots

train_set.plot(kind='density', subplots=True, layout=(8,7), sharex=False, sharey=False, figsize=[30, 30])
pyplot.show()

In [ ]:
# Box and Whisker Plots

train_set.plot(kind='box', subplots=True, layout=(8,7), sharex=False, sharey=False,figsize=[30, 30])
pyplot.show()

##### Data is extremely skewed and there are extreme outliers, thus, data requires pre-processing.

In [ ]:
# Search for correlations of numeric features

correlations = train_set[numeric_cols].corr(method='pearson')

correlations

In [ ]:
# Correlation Matrix Plot

fig = pyplot.figure(figsize=[30, 30])
ax = fig.add_subplot(111)
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)

ticks = np.arange(len(correlations.columns))

ax.set_xticks(ticks)
ax.set_yticks(ticks)

short_names = list(train_set[numeric_cols].columns)

ax.set_xticklabels(short_names)
ax.set_yticklabels(short_names)

pyplot.xticks(rotation=90)     # https://www.geeksforgeeks.org/how-to-rotate-x-axis-tick-label-text-in-matplotlib/?ysclid=lxdkiwmkrh456759424

pyplot.show()

In [ ]:
# Scatter plot Matrix

axes = pd.plotting.scatter_matrix(train_set[numeric_cols], figsize=(20, 20))

for ax_row in axes:
    for ax in ax_row:
        ax.xaxis.label.set_rotation(90)
        ax.yaxis.label.set_rotation(0)
        ax.xaxis.label.set_fontsize(8)
        ax.yaxis.label.set_fontsize(8)
        ax.tick_params(axis='both', labelsize=5)

        # Shift y labels left
        ax.yaxis.set_label_position("left")
        ax.yaxis.label.set_horizontalalignment('right')
        ax.yaxis.label.set_x(-0.2)

pyplot.show()

##### Correlation matrix shows strong correlation (almost 1.0) between 'first deposits' and 'Q1'. Manual analysis reveal that 'first deposits' column is likely to be corrupted, since a lot of samples have the same float value here as in 'Q1', which does not make sense (there should be an integer).
##### There is also a strong correlation between price features (Q1, Q2, Q3 and Q4).

##### Based on results of data analysis on raw data, some pre-processing is required. In the following sections different levels of pre-processing will be tried. At first, minimal pre-processing without transformation will be applied (handling missing values, encode categorical features, clipping extreme outliers). Secondly, heavy skew will be handled using data transformation.

##### Then the following scenarios will be evaluated:
##### - train all models on minimally pre-processed data (without transformation, but with clipped outliers)
##### - train all models on transformed version of data, but without clipping

##### Rescaling will be applied to models that are sensitive to scales.

## Minimal pre-processing of data

#### Based on results of data analysis on raw data, some pre-processing is required in order to run most models. For example, extreme outliers with enormous values result in ValueError during model training. In this section, minimal pre-processing will be applied (handling missing values, encoding categorical features, dealing with extreme outliers) in order to make the dataset suitable for model training.

In [ ]:
# Remove 'first deposits' manually, since it seems to be corrupted (in majority cases it reproduces Q1 figures, which are prices, not deposits count)

for data_set in data_sets:
    data_set.drop(columns=['first deposits'], inplace=True)

train_set

In [ ]:
# Handle missing values (for numeric 'median' is used due to heavy skew, for categorical values 'most frequent' is used)
# SimpleImputer is trained for all numeric and categorical columns (not only for ones with missing values) to be able to handle possible future missing values.

# Based on: Géron, A. "End-to-end Machine Learning Project" in 'Hands-on' machine learning with Scikit-Learn, Keras & Tensorflow. (O'Reilly Media, Inc, 2019) 2nd edition.
# Reference: https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html

from sklearn.impute import SimpleImputer

# Define numeric and categorical columns
numeric_cols = train_set.select_dtypes(include=['number']).columns.tolist()
categorical_cols = train_set.select_dtypes(exclude=['number']).columns.tolist()

# Create inputers and fit to train set
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')
num_imputer.fit(train_set[numeric_cols])
cat_imputer.fit(train_set[categorical_cols])

# Apply to train, test, val sets
for data_set in [train_set, test_set, val_set]:
    data_set[numeric_cols] = num_imputer.transform(data_set[numeric_cols])
    data_set[categorical_cols] = cat_imputer.transform(data_set[categorical_cols])

# Verify it works (no missing values left)
print("\nRemaining missing values per column in train_set:")
print(train_set.isnull().sum())

In [ ]:
# Determine unique categories for 'Blockchain' and 'Blockchain Type' columns

print(data_sets[0]['Blockchain'].unique())
print(data_sets[0]['Blockchain Type'].unique())

In [ ]:
# Encode categorical features ('Blockchain', 'Blockchain Type') to make all features numeric.
# OneHotEncoder was chosen, since the number of categories is small (4 for 'Blockchain' and 3 for 'Blockchain Type'),
# and OneHotEncoder prevents algorithms to treat close values more similar than distant ones.

# Based on: Based on: Géron, A. "End-to-end Machine Learning Project" in 'Hands-on' machine learning with Scikit-Learn, Keras & Tensorflow. (O'Reilly Media, Inc, 2019) 2nd edition.
# Reference: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html

from sklearn.preprocessing import OneHotEncoder

categorical_cols = ['Blockchain', 'Blockchain Type']
cat_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoder.fit(train_set[categorical_cols])

# Apply to each split
for data_set in [train_set, val_set, test_set]:
    cat_array = cat_encoder.transform(data_set[categorical_cols])
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_cols)
    cat_df = pd.DataFrame(cat_array, columns=cat_feature_names, index=data_set.index)

    # Drop original categorical columns
    data_set.drop(columns=categorical_cols, inplace=True)

    # Add encoded columns
    for col in cat_df.columns:
        data_set[col] = cat_df[col]

# Verify it works (all columns are numeric)
print(train_set.dtypes)

In [ ]:
# Determine how many extreme outliers are in dataset
# Several numeric features contains extreme values like 10ˆ40 or 10ˆ60, that cannot be handled by estimators and produce an error (Input X contains infinity or a value too large for dtype('float32')).

# Reference: https://numpy.org/devdocs/reference/generated/numpy.finfo.html

MAX_F32 = np.finfo(np.float32).max

def count_above_f32(df, name):
    arr = df[numeric_cols].to_numpy()
    above = np.abs(arr) > MAX_F32
    n_above = np.count_nonzero(above)
    total = arr.size

    print(f"{name}: values above float32 max: {n_above} ({n_above / total:.6%})")

count_above_f32(train_set, "train_set")
count_above_f32(val_set,   "val_set")
count_above_f32(test_set,  "test_set")

In [ ]:
# Extract feature names (except 'class') for further interpretability

feature_names = train_set.columns.drop("class").tolist()

print(feature_names)

In [ ]:
# Clip extreme values to make data suitable for model training (applied to copies of data, since this step is not relevant
# after transformation (see next section)).
# Clipping is performed for values above float32 datatype capacity.

# Reference: M. Kuhn and K. Johnson, Applied Predictive Modeling. New York, NY, USA: Springer, 2013.

train_minimal = train_set.copy()
val_minimal   = val_set.copy()
test_minimal  = test_set.copy()

numeric_cols_minimal = train_minimal.select_dtypes(include=['number']).columns.tolist()

# Clip everything to float32 capacity
for data_set in [train_minimal, val_minimal, test_minimal]:
    data_set[numeric_cols_minimal] = data_set[numeric_cols_minimal].clip(upper=MAX_F32)

# Verify that no data left that exceeds float32 capacity
for name, data_set in zip(
    ['train_minimal', 'val_minimal', 'test_minimal'],
    [train_minimal, val_minimal, test_minimal]
):
    arr = data_set[numeric_cols_minimal].to_numpy()
    print(f"{name}: any > MAX_F32? {(np.abs(arr) > MAX_F32).any()}")

In [ ]:
# Separate features from the target variable for minimally pre-processed dataset
# Extract X and Y from train, test and validation sets after minimal pre-processing

# Reference: CSM010-2024-APR, Topic 5. 5.12 Hands-on programming demo: evaluating models

X_train_minimal = train_minimal.drop(columns=['class']).values
Y_train_minimal = train_minimal['class'].values

X_test_minimal = test_minimal.drop(columns=['class']).values
Y_test_minimal = test_minimal['class'].values

X_val_minimal = val_minimal.drop(columns=['class']).values
Y_val_minimal = val_minimal['class'].values

## Data transformation

#### Due to extreme skew, some data transformation is conducted in order to improve performance of models. Firstly, log1p transformation has been attempted, resulting in reducing skew to small to moderate values, but several features were still heavily skewed. Thus, PowerTransformer has been applied to find a power that is the most effective in reducing skew for the dataset.

#### Data after this step of pre-processing also will be used for plotting to assess the effect of pre-processing on data.

In [ ]:
# Transform data to deal with skew
# For columns with heavy skew PowerTransformer is applied

# Reference: M. Kuhn and K. Johnson, Applied Predictive Modeling. New York, NY, USA: Springer, 2013.
# Reference: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html

from sklearn.preprocessing import PowerTransformer

skewed_cols = [
    'MaxPrice (Quarter 1)',
    'MaxPrice (Quarter 2)',
    'MaxPrice (Quarter 3)',
    'MaxPrice (Quarter 4)',
    'Total Variance',
    'Variance of holders with more than 1% tokens',
    'Token balance',
    'Token concentration ratio per holder',
    'the number of Transactions',
    'Google results for project title (first day)',
    'Google results for project title (project duration/2)',
    'Google results for project website (first day)',
    'Google results for project website (duration/2)',
    'Google results for project x profile (first days)',
    'Google results for project x profile (duration/2)'
]

# Yeo-Johnson has been chosen, since it handles zeros safely. At this stage, no scaling is applied
pt = PowerTransformer(method='yeo-johnson', standardize=False)
pt.fit(train_set[skewed_cols].astype(float).values)

for data_set in [train_set, val_set, test_set]:
    data_set[skewed_cols] = pt.transform(data_set[skewed_cols].astype(float).values)

# Assess skew after transformation
skew = train_set.skew()
skew

In [ ]:
# Make snapshots of moderately pre-processed data for further experimentation

train_transformed = train_set.copy()
test_transformed  = test_set.copy()
val_transformed   = val_set.copy()

In [ ]:
# Separate features from the target variable for transformed dataset
# Extract X and Y from train, test and validation sets after transformation

# # Reference: CSM010-2024-APR, Topic 5. 5.12 Hands-on programming demo: evaluating models

X_train_transformed = train_transformed.drop(columns=['class']).values
Y_train_transformed = train_transformed['class'].values

X_test_transformed = test_transformed.drop(columns=['class']).values
Y_test_transformed = test_transformed['class'].values

X_val_transformed = val_transformed.drop(columns=['class']).values
Y_val_transformed = val_transformed['class'].values

### Visualisation after data transformation

In [ ]:
# Histograms after handling skew

train_transformed.hist(figsize=[30, 30])
pyplot.show()

In [ ]:
# Density plots after handling skew

train_transformed.plot(kind='density', subplots=True, layout=(8, 7), sharex=False, sharey=False, figsize=[30, 30])
pyplot.show()

In [ ]:
# Box and Whisker Plots after handling skew

train_transformed.plot(kind='box', subplots=True, layout=(8, 7), sharex=False, sharey=False, figsize=[30, 30])
pyplot.show()

In [ ]:
# Search for correlations of numeric features after handling skew

correlations = train_transformed[numeric_cols].corr(method='pearson')

correlations

In [ ]:
# Correlation Matrix Plot after handling skew

fig = pyplot.figure(figsize=[30, 30])
ax = fig.add_subplot(111)
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)

numeric_cols = correlations.columns

ticks = np.arange(len(numeric_cols))

ax.set_xticks(ticks)
ax.set_yticks(ticks)

short_names = list(numeric_cols)

ax.set_xticklabels(short_names)
ax.set_yticklabels(short_names)

pyplot.xticks(rotation=90)     # https://www.geeksforgeeks.org/how-to-rotate-x-axis-tick-label-text-in-matplotlib/?ysclid=lxdkiwmkrh456759424

pyplot.show()

In [ ]:
# Scatter plot Matrix after handling skew

# Reference: https://matplotlib.org/3.10.3/gallery/text_labels_and_annotations/align_ylabels.html

axes = pd.plotting.scatter_matrix(train_transformed[numeric_cols], figsize=(20, 20))

for ax_row in axes:
    for ax in ax_row:
        ax.xaxis.label.set_rotation(90)
        ax.yaxis.label.set_rotation(0)
        ax.xaxis.label.set_fontsize(8)
        ax.yaxis.label.set_fontsize(8)
        ax.tick_params(axis='both', labelsize=5)

        # Shift y labels left
        ax.yaxis.set_label_position("left")
        ax.yaxis.label.set_horizontalalignment('right')
        ax.yaxis.label.set_x(-0.2)

pyplot.show()

## Model specific pre-processing

##### Since models from different families will be trained on data, data requires some additional pre-processing depending on model trained. Some algorithms perform better on rescaled data (e.g. K-Nearest Neighbors or Linear regression), whereas others (like Random Forest) may degrade after rescaling. Model specific pre-processing will be handled in separate pipelines for particular models before training, and then all versions of data (after minimal pre-processing, after transformation and after specific pre-processing) will be used to train models and evaluate performance.

## Training models

##### Preliminary model training shows that some models (SVC, LogisticRegression, LinearDiscriminantAnalysis, KNeighbors) cannot be properly trained on data without scaling, since they are scale-sensitive, and some features in data have different scales, which result in multiple warnings and errors. Thus, these models are always trained only on data after scaling (using StandardScaler or RobustScaler in Pipeline).

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import cross_validate
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Reference: K. Erofeeva, the coursework for CSM010-2024-APR module


# Instantiate models
# For models that require scaling StandardScaler and RobustScaler are applied
# For baseline no specific model tuning is implemented. It will be added after enriching the dataset
def instantiate_models():
    models = []
    models.append(('RandomForest', RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1)))
    models.append(('XGBoost', XGBClassifier(random_state=seed, eval_metric='logloss', n_jobs=-1)))
    models.append(('KNeighbors', Pipeline([
        ('scaler', RobustScaler()),
        ('model', KNeighborsClassifier())
    ])))
    models.append(('SVC', Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(random_state=seed, probability=True))
    ])))
    return models


# Fit a model, make predictions and print classification report and confusion matrix
def train_model_print_reports_and_matrix(name, model, X_train, Y_train, X_test, Y_test):
    model.fit(X_train, Y_train)
    predicted = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    print(f"\n{name} performance on test set:")
    print(classification_report(Y_test, predicted, digits=3))
    print(confusion_matrix(Y_test, predicted))
    print(f"Test ROC AUC: {roc_auc_score(Y_test, proba):.3f}")


# Assess a model with cross-validation
# A secondary measure of assessment, since when shuffling temporal splitting is lost
def assess_with_cross_val(name, model, kfold, X, Y):
    result = cross_validate(model, X, Y, cv=kfold, scoring=('accuracy', 'roc_auc', 'f1'))
    print(f"{name} mean accuracy {result['test_accuracy'].mean():.3f} (+/- {result['test_accuracy'].std():.3f})")
    print(f"{name} mean AUC {result['test_roc_auc'].mean():.3f} (+/- {result['test_roc_auc'].std():.3f})")
    print(f"{name} mean f1 {result['test_f1'].mean():.3f} (+/- {result['test_f1'].std():.3f})\n")


# Train and assess a model using train and test sets
def train_and_assess(name, model, X_train, Y_train, X_test, Y_test):
    print(name)
    train_model_print_reports_and_matrix(name, model, X_train, Y_train, X_test, Y_test)
    print("\nMetrics for the training set: ")
    assess_with_cross_val(name, model, kfold, X_train, Y_train)
    print("\nMetrics for the test set: ")
    assess_with_cross_val(name, model, kfold, X_test, Y_test)

In [ ]:
# Instantiating StratifiedKfold object for future assessment to preserve class balance in folds
# 5 folds are used, since the size of the dataset is quite small

from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(n_splits=5, random_state=seed, shuffle=True)

In [ ]:
# Train and predict on minimally pre-processed dataset (with clipping, but without transformation)
# Evaluate on test set

models = instantiate_models()

print("Results for models trained on data without transformation:\n")
for name, model in models:
    train_and_assess(name, model, X_train_minimal, Y_train_minimal, X_test_minimal, Y_test_minimal)

In [ ]:
# Train and predict on transformed dataset (without clipping)
# Evaluate on test set

print("Results for models trained on transformed data:\n")
for name, model in models:
    train_and_assess(name, model, X_train_transformed, Y_train_transformed, X_test_transformed, Y_test_transformed)

##### Comparison of results shows that Random Forest and XGBoost are two models that perform best both on minimally pre-processed data and on transformed data, and their performance is not influenced by transformation, since they are capable of dealing with skewed data, outliers and extreme values. However, KNeighbours and SVC are very sensitive to transformation: their performance increased significantly on transformed data. Therefore, for all further experiments transformed version of data will be used.

##### LogisticRegression and LinearDiscriminantAnalysis also were initially trained and tested. However, they both produced multiple warnings driven by specific structure of features. A number of experiments were conducted (dropping features with linear or almost linear dependencies, dropping first column while using OneHotEncoder, more aggressive clipping to reduce tales, models tuning), but warnings still persisted. Therefore, they are dropped from further analysis as not suitable for this particular dataset and not consistent with the Project's main aim.

In [ ]:
# Test models on validation set using transformed dataset

# Reference: K. Erofeeva, the coursework for CSM010-2024-APR module

import time

for name, model in models:
    print(name)
    start_time = time.time()
    train_model_print_reports_and_matrix(name, model, X_train_transformed, Y_train_transformed, X_test_transformed, Y_test_transformed)
    end_time = time.time()
    overall_time = end_time - start_time
    print(f"{name}: Time taken: {overall_time:.2f} seconds")
    predicted_val = model.predict(X_val_transformed)
    print("\nPerformance on val set")
    print(classification_report(Y_val_transformed, predicted_val))
    print(confusion_matrix(Y_val_transformed, predicted_val))
    print("\nResults of cross-validation on the val set")
    assess_with_cross_val(name, model, kfold, X_val_transformed, Y_val_transformed)

## Feature importance assessment

##### Feature importance is assessed after candidate models were trained in order to determine what features appear to be most predictive for each model.

##### The primary method applied is permutation importance, since it is model-agnostic, so works for all candidate models and produces reliable results. As a secondary check, feature importance also was assessed using native 'feature_importance_' for tree-based models (Random Forest and XGBoost). Although it may be biased towards categorical variables with larger number of categories, its results can be informative for comparison and explanation.

###### Reference: André Altmann, Laura Toloşi, Oliver Sander, Thomas Lengauer, Permutation importance: a corrected feature importance measure, Bioinformatics, Volume 26, Issue 10, May 2010, Pages 1340–1347

In [ ]:
# Assess feature importance with permutation importance and plot the results

# Reference: A. Khan, A. Ali, J. Khan, F. Ullah and M. Faheem, "Using Permutation-Based Feature Importance for Improved Machine Learning Model Performance at Reduced Costs," in IEEE Access, vol. 13, pp. 36421-36435, 2025
# Reference: https://scikit-learn.org/stable/modules/permutation_importance.html

from sklearn.inspection import permutation_importance

def assess_feature_importance_with_permutation(name, model, X, Y, feature_names, n_repeats=10):
    result = permutation_importance(
        model, X, Y,
        scoring="f1",
        n_repeats=n_repeats,
        random_state=seed,
        n_jobs=-1
    )

    scores = dict(zip(feature_names, result.importances_mean))
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print(name)
    print(sorted_scores)

    importances = pd.Series(result.importances_mean, index=feature_names)
    std = pd.Series(result.importances_std, index=feature_names)

    fig, ax = pyplot.subplots(figsize=(10, 6))
    importances.plot.bar(yerr=std, ax=ax)
    ax.set_title(f"Permutation importance (F1): {name}")
    ax.set_ylabel("Mean decrease in F1")
    fig.tight_layout()
    pyplot.show()

for name, model in models:
    assess_feature_importance_with_permutation(name, model, X_test_transformed, Y_test_transformed, feature_names)

In [ ]:
# Assess feature importance for Random Forest and XGBoost with feature_importances_ property and plot them

# Reference: CSM010-2024-APR, Topic 5. 5.12 Hands-on programming demo: evaluating models

def assess_feature_importance_RF_XGBoost_with_plots(name, model, feature_names):
    scores = dict(zip(feature_names, model.feature_importances_))
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print(name)
    print(sorted_scores)

    importances = pd.Series(model.feature_importances_, index=feature_names)

    fig, ax = pyplot.subplots(figsize=(10, 6))
    importances.plot.bar(ax=ax)
    ax.set_title(f"Feature importances - {name}")
    ax.set_ylabel("Mean decrease in impurity")
    fig.tight_layout()
    pyplot.show()

for name, model in models:
    if name in ("RandomForest", "XGBoost"):
        assess_feature_importance_RF_XGBoost_with_plots(name, model, feature_names)